# Latent–Quantum Triage for Pneumonia Detection with Uncertainty-Aware Decision Support

## Introduction

This notebook implements a complete, paper-oriented experimental pipeline for pneumonia detection from chest X-rays using compact latent representations, hybrid latent–quantum modeling, uncertainty-aware inference, and triage-oriented decision support. The workflow is designed to be reproducible in Google Colab while saving all key research artifacts to Google Drive for direct use in manuscript preparation.

The study proceeds in a structured manner. It first establishes strong classical baselines, then introduces a latent–quantum hybrid model, and finally evaluates uncertainty, calibration, selective prediction, latent-space structure, dimensionality effects, random-projection ablations, and representative case-based triage behavior. This organization supports both empirical validation and scientific interpretation.

All major outputs are written to Google Drive under the `Outputs/` directory, including:
- `Outputs/outputs_summary.txt` for experiment logs and milestone summaries,
- `Outputs/figures/` for publication-ready plots,
- `Outputs/tables/` for result tables and CSV artifacts,
- `Outputs/others/` for model checkpoints and supplementary artifacts.

The notebook is intentionally organized so that every Python cell is preceded by a short explanatory text cell, making the document suitable both as a reproducibility artifact and as a companion research notebook for the paper.


## 1. Environment Setup

This cell installs the required libraries, imports all dependencies, sets the random seed for reproducibility, detects the execution device, mounts Google Drive, creates the output directories, and initializes the summary log file. It ensures that every later result can be saved directly to Google Drive in a consistent folder structure.

In [ ]:

# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================
!pip install -q medmnist==2.2.2 pennylane torch torchvision torchaudio scikit-learn matplotlib pandas

import os, time, math, random
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import pennylane as qml

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount note:', e)

BASE_DIR = Path('/content/drive/MyDrive')
DATA_PATH = BASE_DIR / 'Datasets' / 'Pneumonia' / 'pneumoniamnist.npz'
OUTPUT_DIR = BASE_DIR / 'Outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
OTHER_DIR = OUTPUT_DIR / 'others'
for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, OTHER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUTPUT_DIR / 'outputs_summary.txt'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write('')

def log_header(title: str):
    line = '=' * 100
    msg = f"""\n{line}
{title}
{line}
"""
    print(msg)
    with open(SUMMARY_PATH, 'a', encoding='utf-8') as f:
        f.write(msg)

def append_summary(text: str):
    text = str(text)
    print(text)
    with open(SUMMARY_PATH, 'a', encoding='utf-8') as f:
        f.write(text + '\n')

@contextmanager
def timed_block(name: str):
    start = time.time()
    append_summary(f'[TIMER START] {name}')
    try:
        yield
    finally:
        elapsed = time.time() - start
        append_summary(f'[TIMER END] {name}: {elapsed:.2f} seconds')

log_header('NOTEBOOK START: LATENT-QUANTUM TRIAGE V3')
append_summary(f'Device: {device}')
append_summary(f'Data path: {DATA_PATH}')
append_summary(f'Output directory: {OUTPUT_DIR}')
append_summary(f'Figures directory: {FIG_DIR}')
append_summary(f'Tables directory: {TABLE_DIR}')
append_summary(f'Other artifacts directory: {OTHER_DIR}')


## 2. Data Loading

This cell loads the PneumoniaMNIST dataset from Google Drive, verifies its contents, prepares the train/validation/test splits, and constructs PyTorch data loaders. It also records dataset shapes and class distributions in the summary log so the experimental context is fully documented.

In [ ]:

# ============================================================
# 2. DATA LOADING
# ============================================================
log_header('DATA LOADING')

class NPZMedMNISTDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images.astype(np.float32)
        if self.images.max() > 1.0:
            self.images = self.images / 255.0
        self.labels = labels.astype(np.int64).reshape(-1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.images[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

with timed_block('Load NPZ dataset'):
    data = np.load(DATA_PATH)
    append_summary(f'NPZ keys: {list(data.keys())}')

def to_channel_first(arr):
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    elif arr.ndim == 4 and arr.shape[-1] == 1:
        arr = np.transpose(arr, (0, 3, 1, 2))
    return arr

train_images = to_channel_first(data['train_images'])
train_labels = data['train_labels']
val_images = to_channel_first(data['val_images'])
val_labels = data['val_labels']
test_images = to_channel_first(data['test_images'])
test_labels = data['test_labels']

train_ds = NPZMedMNISTDataset(train_images, train_labels)
val_ds = NPZMedMNISTDataset(val_images, val_labels)
test_ds = NPZMedMNISTDataset(test_images, test_labels)

BATCH_SIZE = 64
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=PIN_MEMORY)

append_summary(f'Train shape: {train_images.shape}, labels: {train_labels.reshape(-1).shape}')
append_summary(f'Val shape:   {val_images.shape}, labels: {val_labels.reshape(-1).shape}')
append_summary(f'Test shape:  {test_images.shape}, labels: {test_labels.reshape(-1).shape}')
append_summary(f'Train label distribution: {np.bincount(train_labels.reshape(-1))[:2].tolist()}')
append_summary(f'Val label distribution:   {np.bincount(val_labels.reshape(-1))[:2].tolist()}')
append_summary(f'Test label distribution:  {np.bincount(test_labels.reshape(-1))[:2].tolist()}')
append_summary(f'BATCH_SIZE={BATCH_SIZE}, PIN_MEMORY={PIN_MEMORY}')


## 3. Utility Functions

This cell defines the helper functions used throughout the notebook for logging, timing, metric computation, plotting, calibration analysis, and artifact saving. These utilities keep the rest of the notebook concise while ensuring that figures, tables, and summary text are written consistently to Google Drive.

In [ ]:

# ============================================================
# 3. UTILITY FUNCTIONS
# ============================================================
log_header('UTILITY FUNCTIONS')

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def multiclass_probs_from_logits(logits):
    return torch.softmax(logits, dim=1)

def binary_positive_probs_from_logits(logits):
    return multiclass_probs_from_logits(logits)[:, 1]

def compute_ece(probs, labels, n_bins=10):
    probs = np.asarray(probs).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    confidences = np.maximum(probs, 1.0 - probs)
    predictions = (probs >= 0.5).astype(int)
    accuracies = (predictions == labels).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i+1]
        mask = (confidences >= left) & ((confidences <= right) if i == n_bins-1 else (confidences < right))
        if mask.sum() > 0:
            ece += mask.mean() * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)

def compute_metrics_from_probs(labels, probs, threshold=0.5):
    labels = np.asarray(labels).reshape(-1)
    probs = np.asarray(probs).reshape(-1)
    preds = (probs >= threshold).astype(int)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / max(1, tn + fp)
    auc = roc_auc_score(labels, probs)
    ece = compute_ece(probs, labels)
    return {'acc': float(acc), 'f1': float(f1), 'precision': float(precision),
            'recall': float(recall), 'specificity': float(specificity), 'auc': float(auc),
            'ece': float(ece), 'confusion_matrix': cm}

def save_confusion_matrix(cm, title, out_path):
    plt.figure(figsize=(4,4))
    plt.imshow(cm, cmap='Blues')
    plt.title(title)
    plt.colorbar()
    plt.xticks([0,1], ['Normal', 'Pneumonia'])
    plt.yticks([0,1], ['Normal', 'Pneumonia'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i,j]), ha='center', va='center')
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    append_summary(f'Saved confusion matrix: {out_path}')

def save_reliability_diagram(labels, probs, title, out_path):
    frac_pos, mean_pred = calibration_curve(labels, probs, n_bins=10, strategy='uniform')
    plt.figure(figsize=(5,5))
    plt.plot([0,1], [0,1], '--')
    plt.plot(mean_pred, frac_pos, marker='o')
    plt.xlabel('Mean predicted probability')
    plt.ylabel('Fraction of positives')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    append_summary(f'Saved reliability diagram: {out_path}')

def save_history_plot(history_df, model_name, out_path):
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train_loss')
    plt.plot(history_df['epoch'], history_df['val_loss'], marker='o', label='val_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'{model_name} Loss'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.subplot(1,2,2)
    plt.plot(history_df['epoch'], history_df['train_acc'], marker='o', label='train_acc')
    plt.plot(history_df['epoch'], history_df['val_acc'], marker='o', label='val_acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title(f'{model_name} Accuracy'); plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    append_summary(f'Saved history plot: {out_path}')


## 4. Model Definitions

This cell defines the encoder backbone, the baseline CNN, the latent-only model, the quantum layer, and the latent–quantum hybrid architecture. It also instantiates the main models and reports their parameter counts, providing a direct view of model complexity and the role of the quantum branch.

In [ ]:

# ============================================================
# 4. MODEL DEFINITIONS
# ============================================================
log_header('MODEL DEFINITIONS')

class EncoderBackbone(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        self.fc_latent = nn.Linear(32 * 7 * 7, latent_dim)

    def encode(self, x):
        return self.fc_latent(self.flatten(self.features(x)))

class BaselineCNN(nn.Module):
    def __init__(self, latent_dim: int = 16, n_classes: int = 2):
        super().__init__()
        self.backbone = EncoderBackbone(latent_dim)
        self.fc_out = nn.Linear(latent_dim, n_classes)
    def forward(self, x):
        z = self.backbone.encode(x)
        return self.fc_out(z), z

class LatentOnlyMLP(nn.Module):
    def __init__(self, latent_dim: int = 16, n_classes: int = 2, dropout_p: float = 0.2):
        super().__init__()
        self.backbone = EncoderBackbone(latent_dim)
        self.head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim), nn.ReLU(), nn.Dropout(dropout_p), nn.Linear(latent_dim, n_classes)
        )
    def forward(self, x):
        z = self.backbone.encode(x)
        return self.head(z), z

N_QUBITS = 4
Q_DEPTH = 1
q_device = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(q_device, interface='torch', diff_method='backprop')
def quantum_circuit(inputs, weights):
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return tuple(qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS))

class QuantumLayer(nn.Module):
    def __init__(self, n_qubits: int = 4, q_depth: int = 1):
        super().__init__()
        self.weights = nn.Parameter(0.01 * torch.randn(q_depth, n_qubits, 3, dtype=torch.float32))
    def forward(self, x):
        outputs = []
        for sample in x:
            out = quantum_circuit(sample, self.weights)
            out = torch.stack([o if isinstance(o, torch.Tensor) else torch.tensor(o, dtype=x.dtype, device=x.device) for o in out])
            outputs.append(out.to(device=x.device, dtype=x.dtype))
        return torch.stack(outputs)

class FixedRandomProjection(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        weight = torch.randn(in_dim, out_dim, dtype=torch.float32) / math.sqrt(in_dim)
        self.register_buffer('weight', weight)
    def forward(self, x):
        return x @ self.weight

class LatentRandomProjectionHybrid(nn.Module):
    def __init__(self, latent_dim: int = 16, rp_input_dim: int = 4, n_classes: int = 2, dropout_p: float = 0.3):
        super().__init__()
        self.rp_input_dim = rp_input_dim
        self.backbone = EncoderBackbone(latent_dim)
        self.random_proj = FixedRandomProjection(rp_input_dim, rp_input_dim)
        self.head = nn.Sequential(
            nn.Linear(latent_dim + rp_input_dim, latent_dim), nn.ReLU(), nn.Dropout(dropout_p), nn.Linear(latent_dim, n_classes)
        )
    def forward(self, x):
        z = self.backbone.encode(x)
        rp_input = torch.tanh(z[:, :self.rp_input_dim]) * (math.pi / 2.0)
        rp_feats = self.random_proj(rp_input).to(device=z.device, dtype=z.dtype)
        fused = torch.cat([z, rp_feats], dim=1)
        return self.head(fused), z, rp_feats

class LatentQuantumHybrid(nn.Module):
    def __init__(self, latent_dim: int = 16, q_input_dim: int = 4, n_classes: int = 2, dropout_p: float = 0.3):
        super().__init__()
        assert q_input_dim == N_QUBITS
        self.q_input_dim = q_input_dim
        self.backbone = EncoderBackbone(latent_dim)
        self.quantum = QuantumLayer(q_input_dim, Q_DEPTH)
        self.head = nn.Sequential(
            nn.Linear(latent_dim + q_input_dim, latent_dim), nn.ReLU(), nn.Dropout(dropout_p), nn.Linear(latent_dim, n_classes)
        )
    def forward(self, x):
        z = self.backbone.encode(x)
        q_input = torch.tanh(z[:, :self.q_input_dim]) * (math.pi / 2.0)
        q_feats = self.quantum(q_input).to(device=z.device, dtype=z.dtype)
        fused = torch.cat([z, q_feats], dim=1)
        return self.head(fused), z, q_feats

LATENT_DIM = 16
baseline_model = BaselineCNN(latent_dim=LATENT_DIM).to(device)
latent_model = LatentOnlyMLP(latent_dim=LATENT_DIM).to(device)
hybrid_model = LatentQuantumHybrid(latent_dim=LATENT_DIM, q_input_dim=N_QUBITS).to(device)

append_summary(f'BaselineCNN params: {count_parameters(baseline_model):,}')
append_summary(f'LatentOnlyMLP params: {count_parameters(latent_model):,}')
append_summary(f'LatentQuantumHybrid params: {count_parameters(hybrid_model):,}')


## 5. Training Utilities

This cell provides the reusable training and validation routines, checkpointing logic, learning summaries, and helper functions that support systematic model fitting. It standardizes the optimization workflow across all compared models so that the later results are directly comparable.

In [ ]:

# ============================================================
# 5. TRAINING UTILITIES
# ============================================================
log_header('TRAINING UTILITIES')

criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_on_loader(model, loader):
    model.eval()
    total_loss, total = 0.0, 0
    all_labels, all_probs = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)[0]
        loss = criterion(logits, y)
        probs = binary_positive_probs_from_logits(logits)
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
        all_labels.extend(y.cpu().numpy().tolist())
        all_probs.extend(probs.cpu().numpy().tolist())
    metrics = compute_metrics_from_probs(np.array(all_labels), np.array(all_probs))
    metrics['loss'] = float(total_loss / max(1, total))
    return metrics

def fit_model(model, model_name, train_loader, val_loader, epochs=5, lr=1e-3):
    append_summary(f'\nTraining model: {model_name}')
    append_summary(f'Parameter count: {count_parameters(model):,}')
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_val_auc = -np.inf
    history = []
    ckpt_path = OTHER_DIR / f'{model_name}_best.pt'
    for epoch in range(1, epochs + 1):
        start = time.time()
        model.train()
        running_loss, running_correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)[0]
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            preds = torch.argmax(torch.softmax(logits, dim=1), dim=1)
            running_loss += loss.item() * x.size(0)
            running_correct += (preds == y).sum().item()
            total += x.size(0)
        train_loss = running_loss / max(1, total)
        train_acc = running_correct / max(1, total)
        val_metrics = evaluate_on_loader(model, val_loader)
        elapsed = time.time() - start
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                        'val_loss': val_metrics['loss'], 'val_acc': val_metrics['acc'],
                        'val_auc': val_metrics['auc'], 'val_ece': val_metrics['ece'],
                        'epoch_time_sec': elapsed})
        append_summary(
            f"{model_name} | Epoch {epoch} | train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
            f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['acc']:.4f}, val_auc={val_metrics['auc']:.4f}, "
            f"val_ece={val_metrics['ece']:.4f}, time={elapsed:.2f}s"
        )
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            torch.save(model.state_dict(), ckpt_path)
    history_df = pd.DataFrame(history)
    hist_csv = TABLE_DIR / f'{model_name}_history.csv'
    history_df.to_csv(hist_csv, index=False)
    append_summary(f'Saved history to: {hist_csv}')
    hist_png = FIG_DIR / f'{model_name}_history.png'
    save_history_plot(history_df, model_name, hist_png)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    append_summary(f'Loaded best checkpoint from: {ckpt_path}')
    return model, history_df, ckpt_path

def evaluate_model(model, loader, model_name='model', save_artifacts=True):
    metrics = evaluate_on_loader(model, loader)
    if save_artifacts:
        cm_png = FIG_DIR / f'{model_name}_confusion_matrix.png'
        save_confusion_matrix(metrics['confusion_matrix'], f'Confusion Matrix: {model_name}', cm_png)
        all_labels, all_probs = [], []
        with torch.no_grad():
            model.eval()
            for x, y in loader:
                x = x.to(device)
                probs = binary_positive_probs_from_logits(model(x)[0])
                all_labels.extend(y.numpy().tolist())
                all_probs.extend(probs.cpu().numpy().tolist())
        rel_png = FIG_DIR / f'{model_name}_reliability.png'
        save_reliability_diagram(np.array(all_labels), np.array(all_probs), f'Reliability: {model_name}', rel_png)
    append_summary(
        f"TEST | {model_name} | acc={metrics['acc']:.4f}, auc={metrics['auc']:.4f}, f1={metrics['f1']:.4f}, "
        f"precision={metrics['precision']:.4f}, recall={metrics['recall']:.4f}, specificity={metrics['specificity']:.4f}, ece={metrics['ece']:.4f}"
    )
    return metrics


## 6. Train Models

This cell trains the baseline CNN, latent-only model, and latent–quantum hybrid model. It records epoch-level losses, accuracy, AUC, calibration error, training time, checkpoints, and history plots, producing the core training artifacts for the paper.

In [ ]:

# ============================================================
# 6. TRAIN MODELS
# ============================================================
LATENT_DIM = 16
EPOCHS = 5
LR = 1e-3

log_header('MODEL TRAINING')

with timed_block('Train baseline CNN'):
    baseline_model = BaselineCNN(latent_dim=LATENT_DIM).to(device)
    baseline_model, baseline_hist, baseline_ckpt = fit_model(baseline_model, 'baseline_cnn', train_loader, val_loader, epochs=EPOCHS, lr=LR)

with timed_block('Train latent-only model'):
    latent_model = LatentOnlyMLP(latent_dim=LATENT_DIM).to(device)
    latent_model, latent_hist, latent_ckpt = fit_model(latent_model, 'latent_only', train_loader, val_loader, epochs=EPOCHS, lr=LR)

with timed_block('Train latent-quantum hybrid model'):
    hybrid_model = LatentQuantumHybrid(latent_dim=LATENT_DIM, q_input_dim=N_QUBITS).to(device)
    hybrid_model, hybrid_hist, hybrid_ckpt = fit_model(hybrid_model, 'latent_quantum_hybrid', train_loader, val_loader, epochs=EPOCHS, lr=LR)


## 7. Test Evaluation

This cell evaluates the trained models on the held-out test set and saves the main quantitative outputs needed for the paper: confusion matrices, reliability diagrams, and a consolidated test comparison table. It provides the primary benchmark evidence for baseline, latent-only, and hybrid performance.

In [ ]:

# ============================================================
# 7. TEST EVALUATION
# ============================================================
log_header('TEST EVALUATION')

test_rows = []
for model_name, model in [('baseline_cnn', baseline_model), ('latent_only', latent_model), ('latent_quantum_hybrid', hybrid_model)]:
    with timed_block(f'Test inference: {model_name}'):
        metrics = evaluate_model(model, test_loader, model_name=model_name, save_artifacts=True)
    test_rows.append({'model': model_name, 'acc': metrics['acc'], 'f1': metrics['f1'], 'precision': metrics['precision'],
                      'recall': metrics['recall'], 'specificity': metrics['specificity'], 'auc': metrics['auc'], 'ece': metrics['ece']})
test_comparison_df = pd.DataFrame(test_rows).sort_values('acc', ascending=False).reset_index(drop=True)
test_comparison_csv = TABLE_DIR / 'model_test_comparison.csv'
test_comparison_df.to_csv(test_comparison_csv, index=False)
append_summary(f'Saved test comparison table: {test_comparison_csv}')
append_summary(test_comparison_df.to_string(index=False))


## 8. Latent Space Analysis

This cell extracts latent features and studies their structure using PCA and t-SNE. The resulting visualizations help assess whether the learned compact representation meaningfully separates pneumonia and normal cases, which is central to the latent-representation argument of the paper.

In [ ]:

# ============================================================
# 8. LATENT SPACE ANALYSIS
# ============================================================
log_header('LATENT SPACE ANALYSIS')

@torch.no_grad()
def extract_latent_features(model, loader):
    model.eval()
    feats, labels = [], []
    for x, y in loader:
        x = x.to(device)
        z = model(x)[1]
        feats.append(z.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats, axis=0), np.concatenate(labels, axis=0).reshape(-1)

latent_feats, latent_labels = extract_latent_features(latent_model, test_loader)
append_summary(f'Latent feature matrix shape: {latent_feats.shape}')

with timed_block('PCA on latent space'):
    pca = PCA(n_components=2, random_state=SEED)
    latent_pca = pca.fit_transform(latent_feats)
    append_summary(f'PCA explained variance ratio sum (2D): {pca.explained_variance_ratio_.sum():.4f}')
    plt.figure(figsize=(6,5))
    plt.scatter(latent_pca[:,0], latent_pca[:,1], c=latent_labels, cmap='coolwarm', s=12, alpha=0.7)
    plt.title('Latent Space PCA (test set)')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.tight_layout()
    pca_path = FIG_DIR / 'latent_space_pca.png'
    plt.savefig(pca_path, dpi=300, bbox_inches='tight'); plt.close()
    append_summary(f'Saved PCA latent plot: {pca_path}')

with timed_block('t-SNE on latent space'):
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, init='pca', learning_rate='auto')
    latent_tsne = tsne.fit_transform(latent_feats)
    plt.figure(figsize=(6,5))
    plt.scatter(latent_tsne[:,0], latent_tsne[:,1], c=latent_labels, cmap='coolwarm', s=12, alpha=0.7)
    plt.title('Latent Space t-SNE (test set)')
    plt.xlabel('Dim 1'); plt.ylabel('Dim 2'); plt.tight_layout()
    tsne_path = FIG_DIR / 'latent_space_tsne.png'
    plt.savefig(tsne_path, dpi=300, bbox_inches='tight'); plt.close()
    append_summary(f'Saved t-SNE latent plot: {tsne_path}')


## 9. Uncertainty and Triage Analysis

This cell applies Monte Carlo dropout to the hybrid model in order to estimate predictive uncertainty. It then derives triage categories such as confident pneumonia, confident normal, and uncertain refer, and saves both the triage table and supporting visual outputs for decision-oriented analysis.

In [ ]:

# ============================================================
# 9. UNCERTAINTY AND TRIAGE ANALYSIS
# ============================================================
log_header('UNCERTAINTY AND TRIAGE ANALYSIS')

def enable_mc_dropout(model):
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

@torch.no_grad()
def mc_dropout_predict(model, loader, n_passes=20):
    model.eval()
    enable_mc_dropout(model)
    all_mean_probs, all_var_probs, all_entropy, all_labels = [], [], [], []
    for x, y in loader:
        x = x.to(device)
        probs_list = []
        for _ in range(n_passes):
            probs_list.append(binary_positive_probs_from_logits(model(x)[0]).unsqueeze(0))
        probs_stack = torch.cat(probs_list, dim=0)
        mean_probs = probs_stack.mean(dim=0)
        var_probs = probs_stack.var(dim=0)
        entropy = -(mean_probs * torch.log(mean_probs.clamp(min=1e-8)) + (1.0 - mean_probs) * torch.log((1.0 - mean_probs).clamp(min=1e-8)))
        all_mean_probs.extend(mean_probs.cpu().numpy().tolist())
        all_var_probs.extend(var_probs.cpu().numpy().tolist())
        all_entropy.extend(entropy.cpu().numpy().tolist())
        all_labels.extend(y.numpy().tolist())
    return {'labels': np.array(all_labels), 'mean_probs': np.array(all_mean_probs), 'var_probs': np.array(all_var_probs), 'entropy': np.array(all_entropy)}

with timed_block('Hybrid MC-dropout inference'):
    mc_results = mc_dropout_predict(hybrid_model, test_loader, n_passes=20)

mc_metrics = compute_metrics_from_probs(mc_results['labels'], mc_results['mean_probs'])
append_summary(f"HYBRID MC-DROPOUT | acc={mc_metrics['acc']:.4f}, auc={mc_metrics['auc']:.4f}, f1={mc_metrics['f1']:.4f}, ece={mc_metrics['ece']:.4f}, mean_entropy={mc_results['entropy'].mean():.4f}, mean_variance={mc_results['var_probs'].mean():.6f}")

entropy_threshold = np.quantile(mc_results['entropy'], 0.75)
pred_labels = (mc_results['mean_probs'] >= 0.5).astype(int)
triage_labels = ['uncertain_refer' if e >= entropy_threshold else ('confident_pneumonia' if p == 1 else 'confident_normal') for p, e in zip(pred_labels, mc_results['entropy'])]

triage_df = pd.DataFrame({'true_label': mc_results['labels'], 'pred_prob': mc_results['mean_probs'], 'pred_label': pred_labels,
                          'entropy': mc_results['entropy'], 'variance': mc_results['var_probs'], 'triage': triage_labels})
triage_csv = TABLE_DIR / 'hybrid_triage_predictions.csv'
triage_df.to_csv(triage_csv, index=False)
append_summary(f'Saved triage predictions: {triage_csv}')
append_summary('Triage counts:')
append_summary(triage_df['triage'].value_counts().to_string())

covered_mask = triage_df['triage'] != 'uncertain_refer'
coverage = covered_mask.mean()
covered_acc = accuracy_score(triage_df.loc[covered_mask, 'true_label'], triage_df.loc[covered_mask, 'pred_label'])
append_summary(f'Triage coverage={coverage:.4f}, covered-case accuracy={covered_acc:.4f}')

plt.figure(figsize=(6,4))
plt.hist(mc_results['entropy'], bins=20)
plt.xlabel('Predictive entropy'); plt.ylabel('Count'); plt.title('Hybrid MC-Dropout Entropy Histogram')
entropy_hist_path = FIG_DIR / 'hybrid_entropy_hist.png'
plt.tight_layout(); plt.savefig(entropy_hist_path, dpi=300, bbox_inches='tight'); plt.close()
append_summary(f'Saved entropy histogram: {entropy_hist_path}')


## 10. Temperature Scaling

This cell fits temperature scaling on the validation set and applies it to the test predictions for post-hoc calibration analysis. It documents whether calibration improves or degrades after scaling and saves the resulting comparison table to Google Drive.

In [ ]:

# ============================================================
# 10. TEMPERATURE SCALING
# ============================================================
log_header('TEMPERATURE SCALING')

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))
    def forward(self, logits):
        temperature = torch.exp(self.log_temperature).clamp(min=1e-3, max=100.0)
        return logits / temperature

def collect_logits_and_labels(model, loader):
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits_list.append(model(x)[0].detach())
            labels_list.append(y.to(device))
    return torch.cat(logits_list, dim=0), torch.cat(labels_list, dim=0)

def fit_temperature_scaler(model, val_loader, max_iter=200):
    logits, labels = collect_logits_and_labels(model, val_loader)
    scaler = TemperatureScaler().to(device)
    optimizer = optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=max_iter)
    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(scaler(logits), labels)
        loss.backward()
        return loss
    optimizer.step(closure)
    return scaler

def evaluate_with_temperature(model, scaler, loader, model_name):
    all_labels, all_probs = [], []
    with torch.no_grad():
        model.eval()
        for x, y in loader:
            x = x.to(device)
            probs = binary_positive_probs_from_logits(scaler(model(x)[0]))
            all_labels.extend(y.numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    metrics = compute_metrics_from_probs(np.array(all_labels), np.array(all_probs))
    append_summary(f"TEMP-SCALED | {model_name} | acc={metrics['acc']:.4f}, auc={metrics['auc']:.4f}, f1={metrics['f1']:.4f}, ece={metrics['ece']:.4f}")
    return metrics, np.array(all_labels), np.array(all_probs)

temp_rows = []
temp_artifacts = {}
for model_name, model in [('baseline_cnn', baseline_model), ('latent_only', latent_model), ('latent_quantum_hybrid', hybrid_model)]:
    with timed_block(f'Fit temperature scaler: {model_name}'):
        scaler = fit_temperature_scaler(model, val_loader)
    metrics, labels_ts, probs_ts = evaluate_with_temperature(model, scaler, test_loader, model_name)
    temp_rows.append({'model': f'{model_name}_temp_scaled', 'acc': metrics['acc'], 'auc': metrics['auc'], 'f1': metrics['f1'],
                      'precision': metrics['precision'], 'recall': metrics['recall'], 'specificity': metrics['specificity'], 'ece': metrics['ece']})
    temp_artifacts[model_name] = {'labels': labels_ts, 'probs': probs_ts}
temp_df = pd.DataFrame(temp_rows)
temp_csv = TABLE_DIR / 'temperature_scaled_test_results.csv'
temp_df.to_csv(temp_csv, index=False)
append_summary(f'Saved temperature-scaled results: {temp_csv}')
append_summary(temp_df.to_string(index=False))


## 11. Coverage–Risk Curves

This cell generates coverage–accuracy and coverage–risk curves for selective prediction analysis. These plots are especially important for the paper because they show how reliability changes when the system defers uncertain cases rather than forcing a decision.

In [ ]:

# ============================================================
# 11. COVERAGE-RISK CURVES
# ============================================================
log_header('COVERAGE-RISK CURVES')

def coverage_risk_curve(labels, probs, uncertainties=None, n_points=20):
    labels = np.asarray(labels).reshape(-1)
    probs = np.asarray(probs).reshape(-1)
    preds = (probs >= 0.5).astype(int)
    if uncertainties is None:
        confidence = np.maximum(probs, 1.0 - probs)
        order = np.argsort(-confidence)
    else:
        order = np.argsort(np.asarray(uncertainties).reshape(-1))
    labels_sorted = labels[order]
    preds_sorted = preds[order]
    coverages, accuracies, risks = [], [], []
    n = len(labels)
    for frac in np.linspace(0.1, 1.0, n_points):
        k = max(1, int(round(frac * n)))
        yk = labels_sorted[:k]
        pk = preds_sorted[:k]
        acc = accuracy_score(yk, pk)
        coverages.append(k / n)
        accuracies.append(acc)
        risks.append(1.0 - acc)
    return np.array(coverages), np.array(accuracies), np.array(risks)

coverage_artifacts = {}
for model_name, model in [('latent_only', latent_model), ('latent_quantum_hybrid', hybrid_model)]:
    all_labels, all_probs = [], []
    with torch.no_grad():
        model.eval()
        for x, y in test_loader:
            x = x.to(device)
            probs = binary_positive_probs_from_logits(model(x)[0])
            all_labels.extend(y.numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    coverage_artifacts[model_name] = {'labels': np.array(all_labels), 'probs': np.array(all_probs), 'uncertainties': None}
coverage_artifacts['latent_quantum_hybrid_mc_dropout'] = {'labels': mc_results['labels'], 'probs': mc_results['mean_probs'], 'uncertainties': mc_results['entropy']}
for k, v in temp_artifacts.items():
    coverage_artifacts[f'{k}_temp_scaled'] = {'labels': v['labels'], 'probs': v['probs'], 'uncertainties': None}

curve_rows = []
plt.figure(figsize=(7,5))
for name, art in coverage_artifacts.items():
    cov, acc, risk = coverage_risk_curve(art['labels'], art['probs'], art['uncertainties'])
    plt.plot(cov, acc, marker='o', label=name)
    for c, a, r in zip(cov, acc, risk):
        curve_rows.append({'model': name, 'coverage': c, 'accuracy': a, 'risk': r})
plt.xlabel('Coverage'); plt.ylabel('Accuracy on covered cases'); plt.title('Coverage–Accuracy Curve'); plt.grid(True, alpha=0.3); plt.legend(fontsize=8)
cov_acc_path = FIG_DIR / 'coverage_accuracy_curve.png'
plt.tight_layout(); plt.savefig(cov_acc_path, dpi=300, bbox_inches='tight'); plt.close()
append_summary(f'Saved coverage–accuracy curve: {cov_acc_path}')

plt.figure(figsize=(7,5))
for name, art in coverage_artifacts.items():
    cov, acc, risk = coverage_risk_curve(art['labels'], art['probs'], art['uncertainties'])
    plt.plot(cov, risk, marker='o', label=name)
plt.xlabel('Coverage'); plt.ylabel('Risk (1 - accuracy)'); plt.title('Coverage–Risk Curve'); plt.grid(True, alpha=0.3); plt.legend(fontsize=8)
cov_risk_path = FIG_DIR / 'coverage_risk_curve.png'
plt.tight_layout(); plt.savefig(cov_risk_path, dpi=300, bbox_inches='tight'); plt.close()
append_summary(f'Saved coverage–risk curve: {cov_risk_path}')

curve_df = pd.DataFrame(curve_rows)
curve_csv = TABLE_DIR / 'coverage_risk_curve_data.csv'
curve_df.to_csv(curve_csv, index=False)
append_summary(f'Saved curve data: {curve_csv}')


## 12. Ablation Helpers

This cell defines helper classes and functions for the latent dimensionality study and the quantum ablation experiments. It supports systematic comparisons across latent sizes and between latent-only, random-projection, and latent–quantum variants.

In [ ]:

# ============================================================
# 12. ABLATION HELPERS
# ============================================================
log_header('ABLATION HELPERS')

def collect_test_metrics(model, model_name, loader):
    metrics = evaluate_model(model, loader, model_name=model_name, save_artifacts=False)
    return {'model': model_name, 'acc': metrics['acc'], 'auc': metrics['auc'], 'f1': metrics['f1'],
            'precision': metrics['precision'], 'recall': metrics['recall'], 'specificity': metrics['specificity'], 'ece': metrics['ece']}

def train_one_variant(model_ctor, model_name, train_loader, val_loader, test_loader, epochs, lr):
    model = model_ctor().to(device)
    n_params = count_parameters(model)
    append_summary(f'Training variant: {model_name} | params={n_params:,}')
    model, hist, ckpt = fit_model(model, model_name, train_loader, val_loader, epochs=epochs, lr=lr)
    test_row = collect_test_metrics(model, model_name, test_loader)
    test_row['params'] = n_params
    return model, hist, ckpt, test_row


## 13. Latent Dimensionality and Quantum Ablation Study

This cell runs the main ablation study over multiple latent dimensions and model variants. It is a key scientific section of the notebook because it tests compactness, scalability, and whether the quantum branch behaves differently from a fixed random-projection control.

In [ ]:

# ============================================================
# 13. LATENT DIMENSIONALITY + QUANTUM ABLATION STUDY
# ============================================================
log_header('LATENT DIMENSIONALITY + QUANTUM ABLATION STUDY')

LATENT_DIMS_STUDY = [8, 16, 32]
ABLATION_EPOCHS_CLASSICAL = 5
ABLATION_EPOCHS_QUANTUM = 3
ABLATION_LR = 1e-3

ablation_rows = []
for latent_dim in LATENT_DIMS_STUDY:
    append_summary(f'\n--- STARTING ABLATION FOR LATENT_DIM={latent_dim} ---')
    with timed_block(f'Ablation latent_only | latent_dim={latent_dim}'):
        _, _, _, row_latent = train_one_variant(lambda ld=latent_dim: LatentOnlyMLP(latent_dim=ld), f'ablation_latent_only_ld{latent_dim}', train_loader, val_loader, test_loader, ABLATION_EPOCHS_CLASSICAL, ABLATION_LR)
        row_latent['latent_dim'] = latent_dim; row_latent['variant'] = 'latent_only'; ablation_rows.append(row_latent)
    with timed_block(f'Ablation latent_random_projection | latent_dim={latent_dim}'):
        _, _, _, row_rp = train_one_variant(lambda ld=latent_dim: LatentRandomProjectionHybrid(latent_dim=ld, rp_input_dim=N_QUBITS), f'ablation_latent_random_projection_ld{latent_dim}', train_loader, val_loader, test_loader, ABLATION_EPOCHS_CLASSICAL, ABLATION_LR)
        row_rp['latent_dim'] = latent_dim; row_rp['variant'] = 'latent_random_projection'; ablation_rows.append(row_rp)
    with timed_block(f'Ablation latent_quantum_hybrid | latent_dim={latent_dim}'):
        _, _, _, row_q = train_one_variant(lambda ld=latent_dim: LatentQuantumHybrid(latent_dim=ld, q_input_dim=N_QUBITS), f'ablation_latent_quantum_hybrid_ld{latent_dim}', train_loader, val_loader, test_loader, ABLATION_EPOCHS_QUANTUM, ABLATION_LR)
        row_q['latent_dim'] = latent_dim; row_q['variant'] = 'latent_quantum_hybrid'; ablation_rows.append(row_q)

ablation_df = pd.DataFrame(ablation_rows)[['latent_dim', 'variant', 'model', 'params', 'acc', 'auc', 'f1', 'precision', 'recall', 'specificity', 'ece']].sort_values(['latent_dim', 'variant']).reset_index(drop=True)
ablation_csv = TABLE_DIR / 'latent_dim_quantum_ablation_results.csv'
ablation_df.to_csv(ablation_csv, index=False)
append_summary(f'Saved ablation table: {ablation_csv}')
append_summary(ablation_df.to_string(index=False))


## 14. Ablation Plots

This cell converts the ablation tables into publication-ready plots that summarize how accuracy, AUC, F1, specificity, and calibration vary with latent dimension and model type. These visual summaries are saved to Google Drive for direct inclusion in the paper drafting process.

In [ ]:

# ============================================================
# 14. ABLATION PLOTS
# ============================================================
log_header('ABLATION PLOTS')

for metric in ['acc', 'auc', 'f1', 'specificity', 'ece']:
    plt.figure(figsize=(7,5))
    for variant in ablation_df['variant'].unique():
        sub = ablation_df[ablation_df['variant'] == variant].sort_values('latent_dim')
        plt.plot(sub['latent_dim'], sub[metric], marker='o', label=variant)
    plt.xlabel('Latent dimension'); plt.ylabel(metric.upper()); plt.title(f'{metric.upper()} vs Latent Dimension')
    plt.legend(); plt.grid(True, alpha=0.3)
    out_path = FIG_DIR / f'ablation_{metric}_vs_latent_dim.png'
    plt.savefig(out_path, dpi=300, bbox_inches='tight'); plt.close()
    append_summary(f'Saved ablation plot: {out_path}')


## 15. Best Variant per Latent Dimension

This cell identifies the best-performing variant for each latent dimension according to both accuracy and calibration error. The resulting tables are useful when writing the results section because they clearly separate performance-oriented and reliability-oriented conclusions.

In [ ]:

# ============================================================
# 15. BEST VARIANT PER LATENT DIMENSION
# ============================================================
log_header('BEST VARIANT PER LATENT DIMENSION')

best_acc_df = ablation_df.sort_values(['latent_dim', 'acc'], ascending=[True, False]).groupby('latent_dim').head(1)
best_ece_df = ablation_df.sort_values(['latent_dim', 'ece'], ascending=[True, True]).groupby('latent_dim').head(1)

best_acc_csv = TABLE_DIR / 'best_ablation_by_accuracy.csv'
best_ece_csv = TABLE_DIR / 'best_ablation_by_ece.csv'
best_acc_df.to_csv(best_acc_csv, index=False)
best_ece_df.to_csv(best_ece_csv, index=False)
append_summary('Best variant by ACC per latent dimension:')
append_summary(best_acc_df.to_string(index=False))
append_summary(f'Saved: {best_acc_csv}')
append_summary('Best variant by ECE per latent dimension:')
append_summary(best_ece_df.to_string(index=False))
append_summary(f'Saved: {best_ece_csv}')


## 16. Case-Based Triage Analysis

This cell extracts representative examples of confident correct predictions, uncertain cases, and overconfident errors. It saves both a table and a figure, providing concrete qualitative evidence for the triage-oriented behavior of the uncertainty-aware hybrid model.

In [ ]:

# ============================================================
# 16. CASE-BASED TRIAGE ANALYSIS
# ============================================================
log_header('CASE-BASED TRIAGE ANALYSIS')

test_images_np = test_images.copy()
triage_df = triage_df.copy()
triage_df['index'] = np.arange(len(triage_df))
triage_df['correct'] = (triage_df['true_label'] == triage_df['pred_label']).astype(int)
triage_df['confidence'] = np.maximum(triage_df['pred_prob'], 1.0 - triage_df['pred_prob'])

case_specs = [
    ('confident_correct_pneumonia', (triage_df['triage'] == 'confident_pneumonia') & (triage_df['correct'] == 1), False),
    ('confident_correct_normal', (triage_df['triage'] == 'confident_normal') & (triage_df['correct'] == 1), False),
    ('uncertain_case', triage_df['triage'] == 'uncertain_refer', True),
    ('overconfident_error', (triage_df['correct'] == 0), False),
]

selected_rows = []
for case_name, mask, sort_entropy_desc in case_specs:
    subset = triage_df[mask].copy()
    if len(subset) == 0:
        continue
    subset = subset.sort_values('entropy', ascending=not sort_entropy_desc if False else not False)
    if sort_entropy_desc:
        subset = subset.sort_values('entropy', ascending=False)
    else:
        subset = subset.sort_values('confidence', ascending=False)
    chosen = subset.iloc[0].copy()
    chosen['case_type'] = case_name
    selected_rows.append(chosen)

case_df = pd.DataFrame(selected_rows)
case_csv = TABLE_DIR / 'case_based_triage_analysis.csv'
case_df.to_csv(case_csv, index=False)
append_summary(f'Saved case-based triage table: {case_csv}')
append_summary(case_df.to_string(index=False))

if len(case_df) > 0:
    n = len(case_df)
    plt.figure(figsize=(4 * n, 4))
    for i, row in enumerate(case_df.itertuples(index=False), start=1):
        idx = int(row.index)
        plt.subplot(1, n, i)
        plt.imshow(test_images_np[idx, 0], cmap='gray')
        plt.axis('off')
        plt.title(f"{row.case_type}\ntrue={row.true_label}, pred={row.pred_label}\np={row.pred_prob:.3f}, H={row.entropy:.3f}")
    case_fig = FIG_DIR / 'case_based_triage_examples.png'
    plt.tight_layout(); plt.savefig(case_fig, dpi=300, bbox_inches='tight'); plt.close()
    append_summary(f'Saved case-based triage figure: {case_fig}')


## 17. Final Summary

This cell consolidates the main results into final summary tables and writes the closing experiment log. It acts as the final checkpoint of the notebook, ensuring that all figures, tables, and numerical comparisons needed for the paper are available on Google Drive.

In [ ]:

# ============================================================
# 17. FINAL SUMMARY
# ============================================================
log_header('FINAL SUMMARY')

final_rows = test_rows + temp_rows + [{
    'model': 'latent_quantum_hybrid_mc_dropout', 'acc': mc_metrics['acc'], 'auc': mc_metrics['auc'], 'f1': mc_metrics['f1'],
    'precision': mc_metrics['precision'], 'recall': mc_metrics['recall'], 'specificity': mc_metrics['specificity'], 'ece': mc_metrics['ece'],
}]
final_df = pd.DataFrame(final_rows)
final_csv = TABLE_DIR / 'final_summary_table.csv'
final_df.to_csv(final_csv, index=False)
append_summary(f'Saved final summary table: {final_csv}')
append_summary(final_df.to_string(index=False))
append_summary(f'Notebook completed successfully. Review summary log at: {SUMMARY_PATH}')
